In [27]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [28]:
df = pd.read_csv('../data/training_data.csv')
print(df.shape)
print(df.head())

(3017586, 12)
   p1_rating_before  p1_rank  p1_power  p1_overall_wr  p1_char_wr  \
0              1338       23    190164       0.346154    0.346154   
1              1918       27    365620       0.557143    0.557143   
2              1326       16    120528       0.476190    0.476190   
3              1505       24    197332       0.592593    0.592593   
4              1469       23    196946       0.482143    0.575758   

   p2_rating_before  p2_rank  p2_power  p2_overall_wr  p2_char_wr  matchup_wr  \
0              1399       23    191175       0.333333    0.333333    0.494962   
1              1660       26    237387       0.000000    0.000000    0.476821   
2              1265       15    145756       0.272727    0.333333    0.495907   
3              1504       24    232311       0.500000    0.500000    0.540708   
4              1414       23    203489       0.466667    0.259259    0.498515   

   winner  
0       1  
1       1  
2       1  
3       2  
4       1  


In [29]:
# winner=3 means a draw (3-3 rounds) — drop for binary classification
draws = (df['winner'] == 3).sum()
df = df[df['winner'].isin([1, 2])].copy()
print(f'Removed {draws} draws; training on {len(df)} matches')

p1_cols = ['p1_rating_before', 'p1_rank', 'p1_power', 'p1_overall_wr', 'p1_char_wr']
p2_cols = ['p2_rating_before', 'p2_rank', 'p2_power', 'p2_overall_wr', 'p2_char_wr']

X_p1 = df[p1_cols].values
X_p2 = df[p2_cols].values
X_matchup = df[['matchup_wr']].values
y = (df['winner'] == 1).astype(np.float32).values  # 1.0 = P1 won, 0.0 = P2 won

print('X_p1:', X_p1.shape)
print('X_p2:', X_p2.shape)
print('X_matchup:', X_matchup.shape)
print('y:', y.shape, '| unique labels:', sorted(set(y.tolist())))

Removed 391 draws; training on 3017195 matches
X_p1: (3017195, 5)
X_p2: (3017195, 5)
X_matchup: (3017195, 1)
y: (3017195,) | unique labels: [0.0, 1.0]


In [30]:
X_p1_train, X_p1_val, X_p2_train, X_p2_val, X_matchup_train, X_matchup_val, y_train, y_val = train_test_split(
    X_p1, X_p2, X_matchup, y, test_size=0.2, random_state=42
)
print('Train:', X_p1_train.shape, X_p2_train.shape, X_matchup_train.shape)
print('Val:  ', X_p1_val.shape, X_p2_val.shape, X_matchup_val.shape)

Train: (2413756, 5) (2413756, 5) (2413756, 1)
Val:   (603439, 5) (603439, 5) (603439, 1)


In [31]:
# Standardize features (fit on train only to avoid data leakage)
player_scaler = StandardScaler()
player_scaler.fit(np.vstack([X_p1_train, X_p2_train]))

X_p1_train = player_scaler.transform(X_p1_train)
X_p2_train = player_scaler.transform(X_p2_train)
X_p1_val = player_scaler.transform(X_p1_val)
X_p2_val = player_scaler.transform(X_p2_val)

matchup_scaler = StandardScaler()
X_matchup_train = matchup_scaler.fit_transform(X_matchup_train)
X_matchup_val = matchup_scaler.transform(X_matchup_val)

X_p1_train = torch.tensor(X_p1_train, dtype=torch.float32)
X_p1_val = torch.tensor(X_p1_val, dtype=torch.float32)
X_p2_train = torch.tensor(X_p2_train, dtype=torch.float32)
X_p2_val = torch.tensor(X_p2_val, dtype=torch.float32)
X_matchup_train = torch.tensor(X_matchup_train, dtype=torch.float32)
X_matchup_val = torch.tensor(X_matchup_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

print('Features standardized using train-set statistics')

Features standardized using train-set statistics


In [32]:
class PlayerEncoder(nn.Module):
    """Layer 1: encodes one player's 5 features into a 32-dim embedding."""
    def __init__(self, input_dim=5, hidden_dim=64, embed_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x):
        return self.encoder(x)

player_encoder = PlayerEncoder()
print(player_encoder)

# Quick sanity check: same weights used for both players
embed_p1 = player_encoder(X_p1_train[:4])
embed_p2 = player_encoder(X_p2_train[:4])
print('P1 embedding shape:', embed_p1.shape)
print('P2 embedding shape:', embed_p2.shape)

PlayerEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
)
P1 embedding shape: torch.Size([4, 32])
P2 embedding shape: torch.Size([4, 32])


In [33]:
class MatchPredictor(nn.Module):
    """Full model: Layer 1 (PlayerEncoder) + Layer 2 (binary classifier)."""
    def __init__(self, player_input_dim=5, hidden_dim=64, embed_dim=32, classifier_hidden=32):
        super().__init__()
        self.player_encoder = PlayerEncoder(player_input_dim, hidden_dim, embed_dim)
        combined_dim = embed_dim * 2 + 1  # both embeddings + matchup_wr
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, classifier_hidden),
            nn.ReLU(),
            nn.Linear(classifier_hidden, 1),  # raw logit (no sigmoid)
        )

    def forward(self, p1_features, p2_features, matchup_wr):
        embed_p1 = self.player_encoder(p1_features)
        embed_p2 = self.player_encoder(p2_features)
        combined = torch.cat([embed_p1, embed_p2, matchup_wr], dim=1)
        return self.classifier(combined)

model = MatchPredictor()
print(model)

with torch.no_grad():
    sample_logits = model(X_p1_train[:4], X_p2_train[:4], X_matchup_train[:4])
    sample_probs = torch.sigmoid(sample_logits)
print('Output shape:', sample_logits.shape)
print('Sample probabilities:', sample_probs.squeeze().tolist())

MatchPredictor(
  (player_encoder): PlayerEncoder(
    (encoder): Sequential(
      (0): Linear(in_features=5, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=65, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)
Output shape: torch.Size([4, 1])
Sample probabilities: [0.5515233874320984, 0.5396323204040527, 0.5388268828392029, 0.5414883494377136]


In [34]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [35]:
EPOCHS = 5
BATCH_SIZE = 1024

train_dataset = TensorDataset(
    X_p1_train, X_p2_train, X_matchup_train, y_train.unsqueeze(1)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for p1_batch, p2_batch, matchup_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(p1_batch, p2_batch, matchup_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{EPOCHS} — train loss: {avg_loss:.4f}')

Epoch 1/5 — train loss: 0.5690
Epoch 2/5 — train loss: 0.5655
Epoch 3/5 — train loss: 0.5649
Epoch 4/5 — train loss: 0.5647
Epoch 5/5 — train loss: 0.5645


In [36]:
model.eval()
with torch.no_grad():
    val_logits = model(X_p1_val, X_p2_val, X_matchup_val)
    val_probs = torch.sigmoid(val_logits)
    val_loss = criterion(val_logits, y_val.unsqueeze(1))

y_true = y_val.numpy()
y_prob = val_probs.numpy().squeeze()

val_auc = roc_auc_score(y_true, y_prob)
default_acc = ((val_probs > 0.5).float() == y_val.unsqueeze(1)).float().mean().item()

best_threshold = 0.5
best_acc = 0.0
for threshold in np.arange(0.30, 0.71, 0.01):
    preds = (y_prob >= threshold).astype(np.float32)
    acc = (preds == y_true).mean()
    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print(f'Validation loss: {val_loss.item():.4f}')
print(f'Validation AUC: {val_auc:.4f}')
print(f'Validation accuracy @ 0.50 threshold: {default_acc:.4f} ({default_acc * 100:.1f}%)')
print(f'Best validation threshold: {best_threshold:.2f}')
print(f'Validation accuracy @ best threshold: {best_acc:.4f} ({best_acc * 100:.1f}%)')

Validation loss: 0.5653
Validation AUC: 0.7663
Validation accuracy @ 0.50 threshold: 0.6890 (68.9%)
Best validation threshold: 0.51
Validation accuracy @ best threshold: 0.6893 (68.9%)


In [37]:
import joblib
from pathlib import Path

models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), models_dir / 'match_predictor.pt')
joblib.dump(player_scaler, models_dir / 'player_scaler.joblib')
joblib.dump(matchup_scaler, models_dir / 'matchup_scaler.joblib')
joblib.dump({'best_threshold': best_threshold}, models_dir / 'model_config.joblib')

print(f'Model saved to {(models_dir / "match_predictor.pt").resolve()}')
print(f'Scalers + threshold config saved to {models_dir.resolve()}')

Model saved to C:\Users\izeco\OneDrive\Desktop\SDEV 378\tekken_final\tekken-win-predictor\models\match_predictor.pt
Scalers + threshold config saved to C:\Users\izeco\OneDrive\Desktop\SDEV 378\tekken_final\tekken-win-predictor\models
